# Delta Lake avec PySpark — prise en main

Ce notebook te guide dans les opérations essentielles. Exécute les cellules dans l'ordre.

## 1. Créer une SparkSession configurée pour Delta Lake

La configuration active l'extension Delta et le catalogue Delta. Le helper ajoute les bibliothèques JVM compatibles.

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DeltaLake-Learning")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "/workspace/data/warehouse")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Application:", spark.sparkContext.appName)


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/notebook/.ivy2.5.2/cache
The jars for the packages stored in: /home/notebook/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d6998627-6773-4b8e-9a62-218c82271cf1;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 130ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.0 from central in [default]
	io.delta#delta-storage;4.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       co

Spark: 4.0.0
Application: DeltaLake-Learning


## 2. Créer un DataFrame PySpark

In [16]:
data = [
    (1, "Clavier", "Informatique", 25.0, 12),
    (2, "Souris", "Informatique", 15.0, 30),
    (3, "Écran", "Informatique", 180.0, 7),
    (4, "Bureau", "Mobilier", 220.0, 4),
]

columns = ["id", "produit", "categorie", "prix", "stock"]
df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()


+---+-------+------------+-----+-----+
| id|produit|   categorie| prix|stock|
+---+-------+------------+-----+-----+
|  1|Clavier|Informatique| 25.0|   12|
|  2| Souris|Informatique| 15.0|   30|
|  3|  Écran|Informatique|180.0|    7|
|  4| Bureau|    Mobilier|220.0|    4|
+---+-------+------------+-----+-----+

root
 |-- id: long (nullable = true)
 |-- produit: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: double (nullable = true)
 |-- stock: long (nullable = true)



## 3. Écrire et relire une table Delta

In [17]:
delta_path = "/workspace/data/delta/produits"

(df.write
   .format("delta")
   .mode("overwrite")
   .save(delta_path))

df_delta = spark.read.format("delta").load(delta_path)
df_delta.show()


+---+-------+------------+-----+-----+
| id|produit|   categorie| prix|stock|
+---+-------+------------+-----+-----+
|  1|Clavier|Informatique| 25.0|   12|
|  3|  Écran|Informatique|180.0|    7|
|  2| Souris|Informatique| 15.0|   30|
|  4| Bureau|    Mobilier|220.0|    4|
+---+-------+------------+-----+-----+



## 4. Vérifier les fichiers et l'historique Delta

In [18]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)


+-------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                            |
+-------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-11-03 14:25:53.199|WRITE    |{mode -> Overwrite, partitionBy -> []}                                                                                                                                         |
|3      |2026-11-03 14:14:02.554|MERGE    |{predicate -> ["(id#1403L = id#3551L)"], matchedPredicates -> [{"actionType":

## 5. UPDATE : modifier le stock d'un produit

In [19]:
from pyspark.sql.functions import col, expr

delta_table.update(
    condition=col("id") == 1,
    set={"stock": expr("stock + 5")}
)

delta_table.toDF().orderBy("id").show()


26/11/03 14:25:57 WARN UpdateCommand: Could not validate number of records due to missing statistics.


+---+-------+------------+-----+-----+
| id|produit|   categorie| prix|stock|
+---+-------+------------+-----+-----+
|  1|Clavier|Informatique| 25.0|   17|
|  2| Souris|Informatique| 15.0|   30|
|  3|  Écran|Informatique|180.0|    7|
|  4| Bureau|    Mobilier|220.0|    4|
+---+-------+------------+-----+-----+



## 6. DELETE : supprimer un produit

In [20]:
delta_table.delete(condition=col("id") == 4)
delta_table.toDF().orderBy("id").show()


26/11/03 14:25:59 WARN DeleteCommand: Could not validate number of records due to missing statistics.


+---+-------+------------+-----+-----+
| id|produit|   categorie| prix|stock|
+---+-------+------------+-----+-----+
|  1|Clavier|Informatique| 25.0|   17|
|  2| Souris|Informatique| 15.0|   30|
|  3|  Écran|Informatique|180.0|    7|
+---+-------+------------+-----+-----+



## 7. MERGE : faire un upsert

In [21]:
nouveaux_produits = spark.createDataFrame(
    [
        (2, "Souris sans fil", "Informatique", 22.0, 18),
        (5, "Chaise", "Mobilier", 95.0, 10),
    ],
    columns
)

(delta_table.alias("cible")
 .merge(nouveaux_produits.alias("source"), "cible.id = source.id")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

delta_table.toDF().orderBy("id").show()


26/11/03 14:26:02 WARN MapPartitionsRDD: RDD 143 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+---+---------------+------------+-----+-----+
| id|        produit|   categorie| prix|stock|
+---+---------------+------------+-----+-----+
|  1|        Clavier|Informatique| 25.0|   17|
|  2|Souris sans fil|Informatique| 22.0|   18|
|  3|          Écran|Informatique|180.0|    7|
|  5|         Chaise|    Mobilier| 95.0|   10|
+---+---------------+------------+-----+-----+



## 8. Time Travel : lire une ancienne version

In [22]:
# La version 0 correspond à l'écriture initiale de cette table.
version_0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(delta_path)
)

version_0.orderBy("id").show()


+---+-------+------------+-----+-----+
| id|produit|   categorie| prix|stock|
+---+-------+------------+-----+-----+
|  1|Clavier|Informatique| 25.0|   12|
|  2| Souris|Informatique| 15.0|   30|
|  3|  Écran|Informatique|180.0|    7|
|  4| Bureau|    Mobilier|220.0|    4|
+---+-------+------------+-----+-----+



## 9. Requêter la table avec Spark SQL

In [23]:
spark.read.format("delta").load(delta_path).createOrReplaceTempView("produits")

spark.sql("""
    SELECT categorie, COUNT(*) AS nombre_produits, ROUND(AVG(prix), 2) AS prix_moyen
    FROM produits
    GROUP BY categorie
    ORDER BY nombre_produits DESC
""").show()


+------------+---------------+----------+
|   categorie|nombre_produits|prix_moyen|
+------------+---------------+----------+
|Informatique|              3|     75.67|
|    Mobilier|              1|      95.0|
+------------+---------------+----------+



## 10. Arrêter Spark

À exécuter uniquement lorsque tu as terminé ta session.

In [24]:
# spark.stop()